# Sequential Testing (mSPRT) — Demo

This notebook demonstrates the **mixture Sequential Probability Ratio Test (mSPRT)**,
which allows continuous experiment monitoring without inflating the false positive rate.

**Problem**: If you check a standard t-test repeatedly ("peeking"), the Type I error
inflates far above the nominal α — sometimes exceeding 30% for an α=0.05 test.

**Solution**: mSPRT uses a test martingale that is valid at any stopping time.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from src.stats.sequential import msprt_test, sequential_boundaries

sns.set_theme(style='whitegrid', palette='muted')
rng = np.random.default_rng(2024)

## 1. The Peeking Problem

We simulate 5000 null experiments (no true effect) and check the p-value repeatedly.
The naive approach stops whenever `p < 0.05` — this inflates the false positive rate.

In [ ]:
N_SIMS = 2000
N_MAX = 500          # max observations per group
CHECK_EVERY = 25     # check p-value every 25 observations
ALPHA = 0.05

naive_fp = 0    # naive peeking false positives
msprt_fp = 0    # mSPRT false positives

for sim in range(N_SIMS):
    rng_sim = np.random.default_rng(sim)
    ctrl_pool = rng_sim.normal(0, 1, N_MAX)
    trt_pool  = rng_sim.normal(0, 1, N_MAX)  # null: same distribution

    naive_rejected = False
    msprt_rejected = False

    for n in range(CHECK_EVERY, N_MAX + 1, CHECK_EVERY):
        ctrl = ctrl_pool[:n]
        trt  = trt_pool[:n]

        # Naive: standard t-test p-value
        _, p_naive = stats.ttest_ind(trt, ctrl)
        if p_naive < ALPHA:
            naive_rejected = True
            break  # stop at first rejection

    for n in range(CHECK_EVERY, N_MAX + 1, CHECK_EVERY):
        ctrl = ctrl_pool[:n]
        trt  = trt_pool[:n]
        p_msprt, _ = msprt_test(ctrl, trt, alpha=ALPHA)
        if p_msprt < ALPHA:
            msprt_rejected = True
            break

    if naive_rejected: naive_fp += 1
    if msprt_rejected: msprt_fp += 1

print(f"Checking every {CHECK_EVERY} observations, up to {N_MAX} per group")
print(f"Naive peeking  FPR: {naive_fp / N_SIMS:.3f}  (nominal α={ALPHA}) ← INFLATED")
print(f"mSPRT          FPR: {msprt_fp / N_SIMS:.3f}  (nominal α={ALPHA}) ← CONTROLLED")

## 2. mSPRT Power Under the Alternative

Now we check: when there IS a true effect, how quickly does mSPRT detect it?

In [ ]:
TRUE_EFFECT = 0.4
N_SIMS_ALT = 500

stopping_times_msprt  = []
stopping_times_fixed  = []  # fixed-horizon test at n=500

for sim in range(N_SIMS_ALT):
    rng_sim = np.random.default_rng(sim + 10_000)
    ctrl_pool = rng_sim.normal(0, 1, N_MAX)
    trt_pool  = rng_sim.normal(TRUE_EFFECT, 1, N_MAX)

    for n in range(CHECK_EVERY, N_MAX + 1, CHECK_EVERY):
        p, _ = msprt_test(ctrl_pool[:n], trt_pool[:n], alpha=ALPHA)
        if p < ALPHA:
            stopping_times_msprt.append(n)
            break
    else:
        stopping_times_msprt.append(N_MAX + 1)  # never rejected

    # Fixed horizon: just test once at N_MAX
    _, p_fixed = stats.ttest_ind(trt_pool, ctrl_pool)
    stopping_times_fixed.append(N_MAX if p_fixed < ALPHA else None)

detected_msprt = [t for t in stopping_times_msprt if t <= N_MAX]
power_msprt = len(detected_msprt) / N_SIMS_ALT
power_fixed = sum(1 for t in stopping_times_fixed if t is not None) / N_SIMS_ALT
avg_stop = np.mean(detected_msprt) if detected_msprt else float('nan')

print(f"True effect = {TRUE_EFFECT}, n_max = {N_MAX}")
print(f"mSPRT power:         {power_msprt:.3f}")
print(f"Fixed-horizon power: {power_fixed:.3f}")
print(f"Avg stopping time (mSPRT, when detected): {avg_stop:.0f}")

## 3. Visualizing an mSPRT Trajectory

In [ ]:
def compute_msprt_trajectory(ctrl_pool, trt_pool, check_every=10):
    n_max = len(ctrl_pool)
    ns, pvals, lambdas = [], [], []
    for n in range(check_every, n_max + 1, check_every):
        p, lam = msprt_test(ctrl_pool[:n], trt_pool[:n])
        ns.append(n)
        pvals.append(p)
        lambdas.append(lam)
    return ns, pvals, lambdas

rng_traj = np.random.default_rng(7)

# Null trajectory
ctrl_null = rng_traj.normal(0, 1, 600)
trt_null  = rng_traj.normal(0, 1, 600)
ns_null, pvals_null, lam_null = compute_msprt_trajectory(ctrl_null, trt_null)

# Alternative trajectory (true effect)
ctrl_alt = rng_traj.normal(0, 1, 600)
trt_alt  = rng_traj.normal(0.5, 1, 600)
ns_alt, pvals_alt, lam_alt = compute_msprt_trajectory(ctrl_alt, trt_alt)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# P-values over time
ax = axes[0]
ax.plot(ns_null, pvals_null, label='Null (no effect)', linewidth=1.5, alpha=0.8)
ax.plot(ns_alt,  pvals_alt,  label='Alternative (δ=0.5)', linewidth=1.5, alpha=0.8)
ax.axhline(0.05, color='red', linestyle='--', label='α = 0.05')
ax.set_xlabel('Observations per group', fontsize=11)
ax.set_ylabel('Always-valid p-value', fontsize=11)
ax.set_title('mSPRT p-value over time', fontsize=13)
ax.legend()
ax.set_ylim(0, 1)

# Lambda (likelihood ratio) over time
ax = axes[1]
ax.plot(ns_null, lam_null, label='Null (no effect)', linewidth=1.5, alpha=0.8)
ax.plot(ns_alt,  lam_alt,  label='Alternative (δ=0.5)', linewidth=1.5, alpha=0.8)
ax.axhline(1 / 0.05, color='red', linestyle='--', label='Threshold = 1/α = 20')
ax.set_xlabel('Observations per group', fontsize=11)
ax.set_ylabel('Mixture Likelihood Ratio Λ_t', fontsize=11)
ax.set_title('mSPRT Likelihood Ratio over time', fontsize=13)
ax.legend()
ax.set_yscale('log')

plt.tight_layout()
plt.savefig('../docs/msprt_trajectory.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Sequential Boundaries

The `sequential_boundaries` function computes the critical z-value at each interim look.

In [ ]:
n_looks = list(range(50, 550, 50))
bounds = sequential_boundaries(n_looks, alpha=0.05, tau_sq=1.0)

plt.figure(figsize=(10, 5))
plt.plot(n_looks, bounds, 'o-', linewidth=2, markersize=6)
plt.axhline(1.96, color='red', linestyle='--', label='Fixed-horizon z=1.96 (α=0.05)')
plt.xlabel('Observations per group at each look', fontsize=11)
plt.ylabel('Critical z-value', fontsize=11)
plt.title('mSPRT Sequential Boundaries', fontsize=13)
plt.legend()
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

print("\nBoundary table:")
pd.DataFrame({'n_per_group': n_looks, 'critical_z': [round(b, 3) for b in bounds]})

## 5. Summary

| Method | Can peek? | FPR controlled? | Early stopping? |
|---|---|---|---|
| Standard t-test | ❌ No | ❌ FPR inflated | ❌ No |
| Bonferroni correction | ✅ Yes | ✅ Yes | ⚠️ Conservative |
| **mSPRT** | ✅ Yes | ✅ Yes | ✅ Yes |

mSPRT is the recommended approach when you need to monitor experiments in real-time.